# Avatar Try-On Multimodal Pipeline Eval

Use this notebook in Google Colab to evaluate a Gemini-like pipeline: analyze avatar + garment with an Ollama vision model, build a structured try-on prompt, optionally route/generate with a multimodal Replicate image-edit provider, and export a JSON report. You can keep `RUN_GENERATION = False` for analyzer-only debugging.


## 1. Install Dependencies

Recommended runtime: Colab GPU. Start with one model and one case to avoid long downloads.

In [ ]:
%pip -q install pillow pandas requests matplotlib replicate


## 2. Configure Analyzer and Generation Matrix

Primary analyzer candidate: `qwen2.5vl:7b`. For generation, start with one warm Replicate model, then enable more matrix rows only after the first run looks stable.


In [ ]:
OLLAMA_MODEL = "qwen2.5vl:7b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
INSTALL_OLLAMA = True
PULL_MODEL = True

# End-to-end generation is optional. Keep False while evaluating analyzer quality.
RUN_GENERATION = False
REPLICATE_TIMEOUT_SECONDS = 420

# Enable one paid generation row first. Add/enable rows after the first result is stable.
GENERATION_MATRIX = [
    {
        "run_id": "qwen_multimodal",
        "model": "qwen/qwen-image-edit-2511",
        "model_version": None,
        "input_mapping": "multi_image_edit",
        "enabled": True,
    },
    {
        "run_id": "nano_multimodal",
        "model": "google/nano-banana",
        "model_version": None,
        "input_mapping": "google_nano_banana",
        "enabled": False,
    },
]

REPORT_PATH = "avatar-multimodal-pipeline-eval-report.json"
EVAL_RUBRIC_VERSION = "avatar-creative-preview-rubric-2026-05-30"


## 3. Start Ollama

If this cell fails or is too slow, switch to a smaller model such as `gemma3:4b` and rerun from this point.

In [ ]:
import os
import subprocess
import time
import requests

if INSTALL_OLLAMA:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

server_log = open("ollama-server.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=server_log)

for attempt in range(60):
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        if response.status_code == 200:
            print("Ollama is ready")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama did not start within 120 seconds")

if PULL_MODEL:
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)


## 4. Upload Images

Upload avatar/template images and garment images for enabled cases. Then edit `CASES` so filenames match the uploaded files. Keep optional cases disabled until you have their images ready. You can reuse the same avatar across cases.


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


In [ ]:
CASES = [
    {
        "case_id": "short-sleeve-jersey-001",
        "enabled": True,
        "avatar_path": "avatar-case1.png",
        "garment_path": "garment-case1.png",
        "expected": {
            "garment_region": "upper_body",
            # Sports jerseys may be classified as t_shirt by VLMs.
            # Keep expected as jersey for strict score, but inspect output manually.
            "garment_type": "jersey",
            "sleeve_length": "short_sleeve",
        },
        "review_focus": ["short sleeve preservation", "v-neck", "side panels", "chest logos"],
    },
    {
        "case_id": "case-002",
        "enabled": True,
        "avatar_path": "avatar-case2.png",
        "garment_path": "garment-case2.png",
        "expected": {
            "garment_region": "upper_body",
            "garment_type": "t_shirt",
            "sleeve_length": "short_sleeve",
        },
        "review_focus": ["large graphic/text", "short sleeve preservation", "full-body avatar stability"],
    },
    {
        "case_id": "long-sleeve-003",
        "enabled": False,
        "avatar_path": "avatar-case3.png",
        "garment_path": "garment-case3.png",
        "expected": {
            "garment_region": "upper_body",
            "garment_type": "shirt",
            "sleeve_length": "long_sleeve",
        },
        "review_focus": ["long sleeve coverage", "wrist/hand visibility", "cuffs/collar"],
    },
    {
        "case_id": "lower-body-004",
        "enabled": False,
        "avatar_path": "avatar-case4.png",
        "garment_path": "garment-case4.png",
        "expected": {
            "garment_region": "lower_body",
            "garment_type": "pants",
            "sleeve_length": "unknown",
        },
        "review_focus": ["lower body region only", "waist/hem alignment", "upper body preservation"],
    },
    {
        "case_id": "one-piece-005",
        "enabled": False,
        "avatar_path": "avatar-case5.png",
        "garment_path": "garment-case5.png",
        "expected": {
            "garment_region": "full_body",
            "garment_type": "dress",
            "sleeve_length": "unknown",
        },
        "review_focus": ["full body silhouette", "body proportion stability", "garment framing"],
    },
]


## 5. Analyzer Prompt and Helpers

In [ ]:
import base64
import hashlib
import io
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image

PROMPT = """
You are a virtual try-on planning analyzer. You receive IMAGE 1 as a synthetic avatar or anonymized user reference, and IMAGE 2 as the garment/product reference. Return JSON only. Do not generate an image. Do not include markdown.

Analyze IMAGE 2 carefully and return exactly one JSON object with these keys:
- garment_region: one of upper_body, lower_body, full_body
- garment_type: one of shirt, t_shirt, jersey, jacket, hoodie, pants, shorts, skirt, dress, set, full_outfit, unknown
- sleeve_length: one of sleeveless, short_sleeve, long_sleeve, unknown
- neckline: short phrase such as crew neck, v-neck, collar, unknown
- dominant_colors: list of visible garment colors
- logo_or_text: describe visible logos, patches, numbers, or text; use unknown if none
- pattern: describe stripes, panels, trims, texture, or fabric pattern; use unknown if plain
- risk_notes: list of try-on risks, especially sleeve, arm, logo, text, or background contamination risks
- recommended_avatar_framing: one of upper_body, full_body

Do not choose or recommend a generation provider. The product pipeline always uses its configured Qwen avatar preview generator. Prefer full_body when the garment is lower_body or full_body, or when an upper-body garment must be shown with stable hips and legs.
""".strip()

def image_to_ollama_b64(path, size=768):
    image = Image.open(path).convert("RGB")
    image.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), color=(255, 255, 255))
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    output = io.BytesIO()
    canvas.save(output, format="JPEG", quality=88, optimize=True)
    return base64.b64encode(output.getvalue()).decode("utf-8")

def extract_json_object(text):
    text = str(text).strip()
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        text = match.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1:
            raise
        return json.loads(text[start:end + 1])

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def score_expected(intent, expected):
    return {key: intent.get(key) == value for key, value in expected.items()}

MANUAL_REVIEW_TEMPLATE = {
    "avatar_stability": "",
    "garment_structure_accuracy": "",
    "color_and_material_fidelity": "",
    "logo_text_detail_fidelity": "",
    "preview_usefulness": "",
    "overall": "",
    "notes": "",
}

def blank_manual_review():
    return dict(MANUAL_REVIEW_TEMPLATE)

def enabled_cases():
    return [case for case in CASES if case.get("enabled", True)]

def validate_enabled_case_files(cases):
    missing = []
    for case in cases:
        for field in ["avatar_path", "garment_path"]:
            if not Path(case[field]).exists():
                missing.append(f"{case['case_id']}:{field}={case[field]}")
    if missing:
        raise FileNotFoundError("Missing files for enabled cases: " + ", ".join(missing))


## 6. Run Analyzer Eval


In [ ]:
def analyze_case(case):
    started = time.time()
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {
                "role": "user",
                "content": PROMPT,
                "images": [
                    image_to_ollama_b64(case["avatar_path"]),
                    image_to_ollama_b64(case["garment_path"]),
                ],
            }
        ],
        "stream": False,
        "format": "json",
    }
    response = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    body = response.json()
    content = body.get("message", {}).get("content", "")
    intent = extract_json_object(content)
    return {
        "status": "succeeded",
        "latency_seconds": time.time() - started,
        "model": OLLAMA_MODEL,
        "raw_content": content,
        "intent": intent,
        "field_scores": score_expected(intent, case.get("expected", {})),
    }

report_cases = []
enabled_eval_cases = enabled_cases()
validate_enabled_case_files(enabled_eval_cases)
for case in enabled_eval_cases:
    print("Running", case["case_id"])
    try:
        analyzer_result = analyze_case(case)
    except Exception as exc:
        analyzer_result = {
            "status": "failed",
            "error": str(exc),
            "model": OLLAMA_MODEL,
            "intent": None,
            "field_scores": {},
        }
    report_cases.append({
        "case_id": case["case_id"],
        "input_hashes": {
            "avatar_image_sha256": sha256_file(case["avatar_path"]),
            "product_image_sha256": sha256_file(case["garment_path"]),
        },
        "expected": case.get("expected", {}),
        "review_focus": case.get("review_focus", []),
        "manual_quality_notes": blank_manual_review(),
        "analyzer_results": [analyzer_result],
    })

all_results = [result for case in report_cases for result in case["analyzer_results"]]
report = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "finished_at": datetime.now(timezone.utc).isoformat(),
    "summary": {
        "total_cases": len(report_cases),
        "analyzer_runs_per_case": 1,
        "total_analyzer_runs": len(all_results),
        "succeeded": sum(1 for result in all_results if result["status"] == "succeeded"),
        "failed": sum(1 for result in all_results if result["status"] != "succeeded"),
    },
    "analyzer_config": {
        "model": OLLAMA_MODEL,
        "base_url": OLLAMA_BASE_URL,
    },
    "rubric_version": EVAL_RUBRIC_VERSION,
    "generation_matrix": GENERATION_MATRIX,
    "case_matrix": [
        {
            "case_id": case["case_id"],
            "enabled": case.get("enabled", True),
            "expected": case.get("expected", {}),
            "review_focus": case.get("review_focus", []),
        }
        for case in CASES
    ],
    "cases": report_cases,
}
Path(REPORT_PATH).write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(report["summary"], indent=2, sort_keys=True))
print("Report written to", REPORT_PATH)


## 7. Optional: Run End-to-End Multimodal Try-On

Set `RUN_GENERATION = True` and provide `REPLICATE_API_TOKEN` in Colab secrets or environment. This uses the analyzer intent as the source of truth for prompt/category details, then calls a multimodal image-edit provider with avatar + garment.


In [ ]:
import os
import replicate
from IPython.display import display

def enabled_generation_configs():
    return [config for config in GENERATION_MATRIX if config.get("enabled", True)]

def build_multimodal_tryon_prompt(intent):
    colors = ", ".join(intent.get("dominant_colors") or []) or "unknown"
    risks = "; ".join(intent.get("risk_notes") or []) or "unknown"
    garment_label = intent.get("garment_type") or "garment"
    region = intent.get("garment_region") or "upper_body"
    return (
        "Edit only the first image. Return only the edited first image, not a side-by-side comparison. "
        "Use the first image as the synthetic avatar/base image. Use the second image only as the garment reference. "
        f"Apply the {garment_label} to the avatar in the {region} region. "
        "Keep the avatar body proportions, pose, face, camera framing, and background stable. "
        "Preserve the exact garment identity from the second image. "
        f"Structured garment analysis: garment_region={region}; garment_type={garment_label}; "
        f"sleeve_length={intent.get('sleeve_length', 'unknown')}; neckline={intent.get('neckline', 'unknown')}; "
        f"dominant_colors={colors}; logo_or_text={intent.get('logo_or_text', 'unknown')}; "
        f"pattern={intent.get('pattern', 'unknown')}. "
        "Do not invent a different garment, do not remove logos/text/patches, and do not change sleeve length. "
        f"Known risks to avoid: {risks}."
    )

def create_replicate_file(path):
    return open(path, "rb")

def read_replicate_output(output):
    if hasattr(output, "read"):
        return output.read()
    if hasattr(output, "url"):
        url = output.url() if callable(output.url) else output.url
        return requests.get(url, timeout=60).content
    if isinstance(output, list) and output:
        return read_replicate_output(output[0])
    if isinstance(output, str):
        return requests.get(output, timeout=60).content
    raise ValueError(f"Unsupported Replicate output: {type(output)}")

def build_replicate_inputs(case, prompt, config):
    input_mapping = config["input_mapping"]
    if input_mapping == "multi_image_edit":
        return {
            "image": [create_replicate_file(case["avatar_path"]), create_replicate_file(case["garment_path"])],
            "prompt": prompt,
            "aspect_ratio": "match_input_image",
            "go_fast": True,
            "seed": 42,
            "output_format": "png",
            "output_quality": 95,
        }
    if input_mapping == "google_nano_banana":
        return {
            "image_input": [create_replicate_file(case["avatar_path"]), create_replicate_file(case["garment_path"])],
            "prompt": prompt,
            "aspect_ratio": "match_input_image",
            "output_format": "png",
        }
    raise ValueError(f"Unsupported input_mapping: {input_mapping}")

def create_replicate_prediction(config, inputs):
    client = replicate.Client(api_token=os.environ.get("REPLICATE_API_TOKEN"))
    for attempt in range(1, 6):
        try:
            if config.get("model_version"):
                return client.predictions.create(version=config["model_version"], input=inputs)
            return client.predictions.create(model=config["model"], input=inputs)
        except Exception as exc:
            message = str(exc).lower()
            is_rate_limited = "429" in message or "throttled" in message or "rate limit" in message
            if not is_rate_limited or attempt == 5:
                raise
            wait_seconds = 5 * attempt
            print(f"Replicate rate limited; retrying in {wait_seconds}s (attempt {attempt}/5)")
            time.sleep(wait_seconds)
    raise RuntimeError("Could not create Replicate prediction")

def wait_for_replicate_prediction(prediction):
    started = time.time()
    while prediction.status not in {"succeeded", "failed", "canceled"}:
        elapsed = time.time() - started
        if elapsed > REPLICATE_TIMEOUT_SECONDS:
            raise TimeoutError(f"Prediction timeout after {elapsed:.0f}s")
        print(f"Prediction {prediction.id}: {prediction.status} ({elapsed:.0f}s)")
        time.sleep(2)
        prediction.reload()
    if prediction.status != "succeeded":
        raise RuntimeError(f"Prediction {prediction.status}: {getattr(prediction, 'error', None)}")
    return prediction.output

def run_generation_for_case(case, analyzer_result, config):
    if analyzer_result.get("status") != "succeeded":
        return {"status": "skipped", "run_id": config["run_id"], "error": "analyzer failed", "generated_image_path": None}
    prompt = build_multimodal_tryon_prompt(analyzer_result["intent"])
    started = time.time()
    try:
        model_ref = config.get("model_version") or config["model"]
        inputs = build_replicate_inputs(case, prompt, config)
        prediction = create_replicate_prediction(config, inputs)
        output_bytes = read_replicate_output(wait_for_replicate_prediction(prediction))
        output_path = f"{case['case_id']}-{config['run_id']}.png"
        Path(output_path).write_bytes(output_bytes)
        return {
            "status": "succeeded",
            "error": None,
            "run_id": config["run_id"],
            "model": model_ref,
            "input_mapping": config["input_mapping"],
            "latency_seconds": time.time() - started,
            "generated_image_path": output_path,
            "generated_image_sha256": hashlib.sha256(output_bytes).hexdigest(),
            "prompt": prompt,
        }
    except Exception as exc:
        return {
            "status": "failed",
            "error": str(exc),
            "run_id": config["run_id"],
            "model": config.get("model_version") or config["model"],
            "input_mapping": config["input_mapping"],
            "latency_seconds": time.time() - started,
            "generated_image_path": None,
            "prompt": prompt,
        }

if RUN_GENERATION:
    if not os.environ.get("REPLICATE_API_TOKEN"):
        raise RuntimeError("Set REPLICATE_API_TOKEN before RUN_GENERATION=True")
    generation_configs = enabled_generation_configs()
    if not generation_configs:
        raise RuntimeError("GENERATION_MATRIX has no enabled configs")
    generation_cases = enabled_eval_cases if "enabled_eval_cases" in globals() else enabled_cases()
    for case_report, case in zip(report["cases"], generation_cases):
        generation_results = []
        for analyzer_result in case_report["analyzer_results"]:
            for config in generation_configs:
                generation_results.append(run_generation_for_case(case, analyzer_result, config))
        case_report["generation_results"] = generation_results
    all_generation_results = [r for c in report["cases"] for r in c.get("generation_results", [])]
    report["summary"]["generation_runs_per_case"] = len(generation_configs)
    report["summary"]["total_generation_runs"] = len(all_generation_results)
    report["summary"]["generation_succeeded"] = sum(1 for r in all_generation_results if r["status"] == "succeeded")
    report["summary"]["generation_failed"] = sum(1 for r in all_generation_results if r["status"] == "failed")
    Path(REPORT_PATH).write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
    print(json.dumps(report["summary"], indent=2, sort_keys=True))
else:
    print("RUN_GENERATION is False; analyzer-only report kept.")


## 8. Review Analyzer and Generation Results


In [ ]:
import pandas as pd

rows = []
for case in report["cases"]:
    generation_results = case.get("generation_results", [])
    for result in case["analyzer_results"]:
        intent = result.get("intent") or {}
        matching_generations = generation_results or [{}]
        for generation in matching_generations:
            rows.append({
                "case_id": case["case_id"],
                "review_focus": case.get("review_focus"),
                "analyzer_model": result.get("model"),
                "analyzer_status": result.get("status"),
                "analyzer_latency_seconds": result.get("latency_seconds"),
                "garment_region": intent.get("garment_region"),
                "garment_type": intent.get("garment_type"),
                "sleeve_length": intent.get("sleeve_length"),
                "logo_or_text": intent.get("logo_or_text"),
                "pattern": intent.get("pattern"),
                "field_scores": result.get("field_scores"),
                "run_id": generation.get("run_id"),
                "generation_status": generation.get("status"),
                "generation_latency_seconds": generation.get("latency_seconds"),
                "generated_image_path": generation.get("generated_image_path"),
            })

review_df = pd.DataFrame(rows)
manual_review_df = review_df[["case_id", "run_id", "generated_image_path"]].copy() if not review_df.empty else pd.DataFrame()
for key in MANUAL_REVIEW_TEMPLATE:
    manual_review_df[key] = ""
MANUAL_REVIEW_CSV_PATH = "avatar-multimodal-manual-review-template.csv"
manual_review_df.to_csv(MANUAL_REVIEW_CSV_PATH, index=False)
review_df


In [ ]:
from zipfile import ZipFile

generated_paths = [
    generation.get("generated_image_path")
    for case in report["cases"]
    for generation in case.get("generation_results", [])
    if generation.get("status") == "succeeded" and generation.get("generated_image_path")
]

for path in generated_paths:
    print(path)
    display(Image.open(path))

if generated_paths:
    zip_path = "avatar-multimodal-generated-images.zip"
    with ZipFile(zip_path, "w") as archive:
        for path in generated_paths:
            archive.write(path, arcname=Path(path).name)
    files.download(zip_path)

if "MANUAL_REVIEW_CSV_PATH" in globals() and Path(MANUAL_REVIEW_CSV_PATH).exists():
    files.download(MANUAL_REVIEW_CSV_PATH)

files.download(REPORT_PATH)
